# BoatPhone — the satellite imagery pipeline, start to finish

**What you are reading.** This is the reproducible record of the *optical* half of the
BoatPhone project: how satellite photographs of one patch of ocean were found, screened,
paid for and downloaded. It is written to be read top to bottom by somebody who has never
seen this repository, and it is the companion to `acoustic_pipeline.ipynb` (the underwater
microphone half) and `vessel_detection_pipeline.ipynb` (finding boats in these pictures).

---

## The project in one paragraph

Satellites see every boat in a patch of ocean, but only for the fraction of a second the
satellite is overhead. A hydrophone — an underwater microphone — hears boats continuously,
day and night, but cannot tell you how many there are. BoatPhone uses each instrument to fix
the other's blind spot: the satellite pictures give us *ground truth* ("there were three
boats here at this exact instant") for a handful of moments, and those moments are used to
check a number computed from the sound. If the two agree, the sound can then estimate boat
traffic continuously — including small recreational boats, which carry no radio transponder
and are invisible to conventional vessel tracking.

The listening station is **Folger Deep**, a cabled observatory in Barkley Sound on the west
coast of Vancouver Island, run by Ocean Networks Canada. The satellite is **PlanetScope**, a
fleet of small satellites that photograph most of the Earth daily at about 3 metres per pixel.

## What *this* notebook achieves

It turns "we have a satellite account and a map coordinate" into **26 cloud-free photographs
of a 10 × 10 km box of ocean centred on the hydrophone, spread across six summers**, with a
record of every scene that was considered and rejected and why.

The hard part is not downloading. It is **not wasting the budget**. The account allows 3,000
square kilometres of imagery per month, which divides into exactly 30 photographs of our box,
and the allowance does not roll over. Every step below exists to make sure that a photograph
we pay for is one we can actually use.

## What this notebook does *not* establish

- **It does not find boats.** That is `vessel_detection_pipeline.ipynb`. This notebook only
  decides which pictures are worth having.
- **The billing basis was never measured.** We still do not know for certain whether Planet
  charges for the shape we asked to cut out or for the rectangle it actually delivers. The
  batch size was set to survive either answer. See section 8.
- **Two of the six summer months were never searched.** A hard-coded month range in the
  original notebook searched June–August when the project's shared definition of the season
  is May–September. Roughly a third of the candidate pool was therefore never looked at.
  See section 10.

---
# 0. How to run this notebook

There are **two switches**, and the second one spends real money.

| Switch | Default | What it controls |
|---|---|---|
| `RUN_LIVE` | `False` | Steps that call the Planet servers: the search, the cloud screening, the previews. These cost no imagery budget, but they need an account key and can take hours. |
| `ALLOW_SPENDING` | `False` | The single step that places an order. This draws down the monthly allowance permanently. |

With both switched off, every expensive cell **prints the command or the code it would have
run** and then reads the results that are already saved in the repository, so you can follow
the entire pipeline end to end in about a minute and see the real numbers.

**Where things live.** Two rules, and they are why the code is arranged the way it is:

| Directory | Rule |
|---|---|
| `boatphone/` | The **library**: shared constants and functions. Import from here. |
| `scripts/` | The **entry points**: runnable programs that import from `boatphone/`. |
| `data/` | **Never edited.** Anything computed goes to `data/derived/`. |

The reason for the first rule is worth stating plainly: if the satellite side defined the
study area in one file and the sound side defined it in another, the eventual join between
them would quietly match two *different* areas and produce a wrong answer instead of an error.

In [1]:
# --- Setup. Run this first. ---------------------------------------------------
# Both switches are off. Nothing below will contact Planet or spend anything.
RUN_LIVE       = False    # True to actually run the search / screening / preview steps
ALLOW_SPENDING = False    # True to allow the one cell that places an order

import json, math, os, pathlib, subprocess, sys

# REPO_ROOT is the top of the checkout. This notebook lives in final_notebooks/,
# so the repository root is one directory up.
REPO_ROOT = pathlib.Path.cwd()
if REPO_ROOT.name == "final_notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))   # so `import boatphone` works
os.chdir(REPO_ROOT)                  # so every relative path below is repo-relative
NOTEBOOK_DIR = REPO_ROOT / "final_notebooks"

# Where the acquisition's own working files live. These are small and tracked in git,
# so every number this notebook prints can be reproduced from a fresh clone.
ACQ  = REPO_ROOT / "contributor_folders" / "malachymcc"
WORK = ACQ / "planet_folger"

print("repository root:", REPO_ROOT)
print("acquisition working files:", WORK.relative_to(REPO_ROOT))


# --- Small helpers, shared with the other two notebooks -----------------------
from IPython.display import HTML, display

def rel_to_notebook(path):
    """A path as the notebook's own HTML output needs to see it (e.g. ../data/...).

    Links in notebook output resolve relative to the notebook's folder, not to the
    working directory, so every link needs to climb out of final_notebooks/ first.
    """
    return os.path.relpath(pathlib.Path(path).resolve(), NOTEBOOK_DIR)

def link(path, label=None):
    path = pathlib.Path(path)
    return f'<a href="{rel_to_notebook(path)}" target="_blank">{label or path.name}</a>'

def show_folder(path, label=None):
    """A clickable folder link plus its absolute path, for copy-paste."""
    path = pathlib.Path(path)
    display(HTML(f"&#128193; {link(path, label or str(path))}"))
    print("   absolute path:", path.resolve())

def would_run(description, command_or_code):
    """Show an expensive step without running it.

    Used wherever RUN_LIVE or ALLOW_SPENDING is False. The point is that the reader
    still sees exactly what would happen, rather than a blank cell.
    """
    print(f"NOT RUN — {description}")
    print("-" * 72)
    print(command_or_code.strip())

repository root: /home/jovyan/ohw26_proj_BoatPhone
acquisition working files: contributor_folders/malachymcc/planet_folger


---
# 1. The study area, and why it is that exact size

Everything starts with one square of ocean. It is a **10 × 10 km box centred on the
hydrophone**, and its corners are stored in a single file, `folger_core_aoi.geojson`, which
every later step reads. That single file is deliberate: the search, the cloud check, the
previews and the final cut-out all use the same shape, so a scene is never *chosen* on one
area and *judged* on another.

## The 5-metre shrink, which is not a rounding error

The nominal 10 × 10 km box measures **100.00024 km²** — about 240 square metres *over* a round
100 km². That matters because the monthly allowance is 3,000 km² and Planet's area accounting
is exact: thirty orders of the nominal box would ask for 3,000.007 km² and **the thirtieth
would be refused**.

So every edge is pulled in by 5 metres. The strict minimum needed is 6 millimetres, but 5
metres was chosen to cover a second and larger risk. Our box is a rectangle in
latitude/longitude. On any flat map its sides converge slightly, so it becomes a shape whose
*bounding rectangle* is about 90,000 m² bigger than the shape itself — and a cut-out image
file is always a rectangle. If Planet charges for the delivered rectangle rather than the
requested shape, the rectangle is the number that counts.

At a 5-metre inset both candidates come in under the line:

| What might be charged | Area |
|---|---|
| The requested shape | 99.800 km² |
| Its bounding rectangle | 99.889 km² |
| Bounding rectangle snapped outward a whole pixel | 100.009 km² — *marginally over* |

The cost of the shrink is 5 metres of ocean per edge — less than two pixels — and 0.2% of area.

The cell below re-derives all of this from the file on disk and **asserts** it, so the
notebook cannot quietly drift away from the geometry it claims to use.

In [2]:
from shapely.geometry import shape
from shapely.ops import transform as shapely_transform
from pyproj import Transformer

from boatphone import optical   # the shared library: one definition of the hydrophone

# The hydrophone position. Imported, never re-typed — see the note in section 10 about
# the fact that this is currently the box centre rather than confirmed instrument metadata.
HYD_LON, HYD_LAT = optical.HYDROPHONE_LONLAT

AOI_PATH = ACQ / "folger_core_aoi.geojson"
nominal  = json.load(open(AOI_PATH))
if nominal["type"] == "FeatureCollection":
    nominal = nominal["features"][0]["geometry"]
elif nominal["type"] == "Feature":
    nominal = nominal["geometry"]
assert nominal["type"] == "Polygon", f"expected a Polygon, got {nominal['type']}"

ring = nominal["coordinates"][0]
west, south = min(c[0] for c in ring), min(c[1] for c in ring)
east, north = max(c[0] for c in ring), max(c[1] for c in ring)

# --- the 5 metre shrink, converted from metres into degrees at this latitude ---
AOI_INSET_M = 5.0
dlat = AOI_INSET_M / 111_206.0
dlon = AOI_INSET_M / (111_320.0 * math.cos(math.radians(HYD_LAT)))
AOI_BBOX = (west + dlon, south + dlat, east - dlon, north - dlat)

# `aoi` is now THE study area, in the format Planet's API expects.
aoi = {"type": "Polygon", "coordinates": [[
    [AOI_BBOX[0], AOI_BBOX[1]], [AOI_BBOX[2], AOI_BBOX[1]],
    [AOI_BBOX[2], AOI_BBOX[3]], [AOI_BBOX[0], AOI_BBOX[3]],
    [AOI_BBOX[0], AOI_BBOX[1]]]]}

In [3]:
# Areas must be compared on a map projection where areas are TRUE. Degrees are not a
# unit of area, and a fraction computed in degrees carries enough error at this latitude
# to swamp the margins we are working with. This projection is centred on the hydrophone,
# so the "is the box centred on the instrument?" check below is meaningful.
AOI_LAEA = f"+proj=laea +lat_0={HYD_LAT} +lon_0={HYD_LON} +datum=WGS84 +units=m"
to_metres = Transformer.from_crs("EPSG:4326", AOI_LAEA, always_xy=True).transform
AOI_M     = shapely_transform(to_metres, shape(aoi))

MIN_CHARGE, MONTHLY_QUOTA = 100.0, 3000.0        # km^2 per scene, km^2 per month

AOI_AREA_KM2     = AOI_M.area / 1e6
CHARGE_PER_SCENE = max(AOI_AREA_KM2, MIN_CHARGE)  # the 100 km^2 minimum charge applies
SCENES_PER_MONTH = int(MONTHLY_QUOTA // CHARGE_PER_SCENE)

pb = AOI_M.bounds                                  # the delivered rectangle
AOI_BBOX_KM2 = (pb[2] - pb[0]) * (pb[3] - pb[1]) / 1e6

# Assert the properties the whole study rests on, rather than trusting a constant to
# still match the file on disk. If someone edits the geojson, this cell fails loudly.
offset = math.hypot(AOI_M.centroid.x, AOI_M.centroid.y)
assert offset < 50.0,          f"box centre is {offset:.0f} m from the hydrophone"
assert AOI_AREA_KM2 < 100.0,   f"requested shape is {AOI_AREA_KM2:.6f} km2 — not under the minimum"
assert AOI_BBOX_KM2 < 100.0,   f"delivered rectangle is {AOI_BBOX_KM2:.6f} km2 — not under the minimum"
assert SCENES_PER_MONTH * CHARGE_PER_SCENE <= MONTHLY_QUOTA, "a full month's batch exceeds the quota"

print(f"box extent      : {(pb[2]-pb[0])/1000:.4f} x {(pb[3]-pb[1])/1000:.4f} km")
print(f"centred         : {offset:.1f} m from the hydrophone at {HYD_LON}, {HYD_LAT}")
print(f"requested shape : {AOI_AREA_KM2:.5f} km2   ({(100.0-AOI_AREA_KM2)*1e6:,.0f} m2 under the limit)")
print(f"delivered rect. : {AOI_BBOX_KM2:.5f} km2   ({(100.0-AOI_BBOX_KM2)*1e6:,.0f} m2 under the limit)")
print(f"budget          : {SCENES_PER_MONTH} scenes/month at {CHARGE_PER_SCENE:.2f} km2 each"
      f" = {SCENES_PER_MONTH*CHARGE_PER_SCENE:,.0f} of {MONTHLY_QUOTA:,.0f} km2")

box extent      : 9.9988 x 9.9901 km
centred         : 0.8 m from the hydrophone at -125.278277, 48.8142
requested shape : 99.80015 km2   (199,850 m2 under the limit)
delivered rect. : 99.88922 km2   (110,782 m2 under the limit)
budget          : 30 scenes/month at 100.00 km2 each = 3,000 of 3,000 km2


---
# 2. Two separate budgets

This is the single most important thing to understand before running anything, because the
two budgets behave completely differently.

| Budget | Allowance | Cost of one scene | Ceiling |
|---|---|---|---|
| **Imagery** — the actual photographs | 3,000 km² per month | 100 km² minimum whenever you cut a piece out | **30 photographs a month** |
| **Preview tiles** — small web-map images for looking | about 98,000 tiles per month | 196 tiles to cover our whole box | about 500 previews |

Three things are **free** and do not touch the imagery allowance: the search itself, the small
thumbnail images, and the quality mask that tells you where the clouds are. That is what makes
the design below possible — **everything up to the ordering step costs nothing**, so the
screening can be as thorough as you like, and only the last step spends.

Neither allowance rolls over. An unspent photograph at the end of the month is gone.

---
# 3. Signing in

Your Planet key is read from the environment or from the SDK's saved login. **It is never
written into this notebook and never committed.** If you have not logged in before, run
`planet auth init` in a terminal once.

In [4]:
# This cell is safe to run: it reports whether a key is present, and never prints it.
API_KEY = os.environ.get("PL_API_KEY")
if API_KEY:
    source = "the PL_API_KEY environment variable"
else:
    saved = pathlib.Path.home() / ".planet.json"
    if saved.exists():
        API_KEY, source = json.load(open(saved)).get("key"), "the saved SDK login (~/.planet.json)"
    else:
        source = None

if API_KEY:
    print(f"Planet key found via {source}. Its value is never printed and never committed.")
else:
    print("No Planet key found. Run `planet auth init` in a terminal, or set PL_API_KEY.")
    print("Everything in this notebook that reads saved results will still work without one.")

Planet key found via the saved SDK login (~/.planet.json). Its value is never printed and never committed.


---
# 4. Step one — search, deliberately loose

The search asks Planet for every PlanetScope scene that **touches** our box, in June to
August of 2020–2025, with less than 60% cloud over the whole scene.

Sixty percent sounds far too generous, and that is the point. Planet's cloud figure describes
the **entire scene**, which is about 637 km² — our box is only 15% of it. A scene can be
reported as half-clouded and still be perfectly clear over the water we care about, and the
reverse is just as true. So the search is left loose on purpose, and the real cloud decision
is made in step three on the box itself, using a free mask.

**This returned 552 candidate scenes**, cached to `search_results.json`.

In [5]:
# The search is a live call and takes a minute or two. The result is already cached, so
# with RUN_LIVE = False this cell shows the query and then reads the cached answer.
SEARCH_JSON = WORK / "search_results.json"

SEARCH_CODE = '''
from datetime import datetime
from planet import Auth, Session, data_filter

YEARS, SUMMER, ITEM_TYPE = range(2020, 2026), (6, 9), "PSScene"
SEARCH_CLOUD_MAX = 0.60          # whole-scene cloud; the real screen is step three

search_filter = data_filter.and_filter([
    data_filter.geometry_filter(aoi),                       # touches our box
    data_filter.range_filter("cloud_cover", lte=SEARCH_CLOUD_MAX),
    data_filter.or_filter([                                 # any summer, 2020-2025
        data_filter.date_range_filter("acquired",
                                      gte=datetime(y, SUMMER[0], 1),
                                      lt =datetime(y, SUMMER[1], 1))
        for y in YEARS]),
    data_filter.permission_filter(),                        # only what we may download
    data_filter.string_in_filter("quality_category", ["standard"]),
])

async with Session(auth=Auth.from_key(API_KEY)) as sess:
    results = sess.client("data").search([ITEM_TYPE], search_filter=search_filter, limit=0)
    items = [i async for i in results]

json.dump(items, open(SEARCH_JSON, "w"))
'''

if RUN_LIVE:
    print("Run the code below in a cell of its own — it needs `await` at notebook top level.")
    print(SEARCH_CODE)
else:
    would_run("the live Planet search (free, no imagery budget)", SEARCH_CODE)

items = json.load(open(SEARCH_JSON))
print(f"\n{len(items)} candidate scenes, cached in {SEARCH_JSON.relative_to(REPO_ROOT)}")

NOT RUN — the live Planet search (free, no imagery budget)
------------------------------------------------------------------------
from datetime import datetime
from planet import Auth, Session, data_filter

YEARS, SUMMER, ITEM_TYPE = range(2020, 2026), (6, 9), "PSScene"
SEARCH_CLOUD_MAX = 0.60          # whole-scene cloud; the real screen is step three

search_filter = data_filter.and_filter([
    data_filter.geometry_filter(aoi),                       # touches our box
    data_filter.range_filter("cloud_cover", lte=SEARCH_CLOUD_MAX),
    data_filter.or_filter([                                 # any summer, 2020-2025
        data_filter.date_range_filter("acquired",
                                      gte=datetime(y, SUMMER[0], 1),
                                      lt =datetime(y, SUMMER[1], 1))
        for y in YEARS]),
    data_filter.permission_filter(),                        # only what we may download
    data_filter.string_in_filter("quality_category", ["standard"]),
])

In [6]:
# Flatten the search result into a table. Every later step adds a column to this table;
# nothing is ever dropped silently — each gate prints how many scenes it removed.
import pandas as pd

df = pd.DataFrame([{
    "id":            it["id"],
    "acquired":      pd.to_datetime(it["properties"]["acquired"]),
    "cloud_cover":   it["properties"]["cloud_cover"],       # whole scene, not our box
    "instrument":    it["properties"].get("instrument"),    # PS2.SD (older) or PSB.SD (newer)
    "sun_elevation": it["properties"].get("sun_elevation"),
    "sun_azimuth":   it["properties"].get("sun_azimuth"),
    "view_angle":    it["properties"].get("view_angle"),
    "sat_azimuth":   it["properties"].get("satellite_azimuth"),
} for it in items])
df["date"] = df["acquired"].dt.date
df["year"] = df["acquired"].dt.year
df = df.sort_values("acquired").reset_index(drop=True)

print(df.groupby("year").agg(scenes=("id", "size"), distinct_days=("date", "nunique")))
print(f"\ntotal: {len(df)} scenes over {df['date'].nunique()} distinct days")

      scenes  distinct_days
year                       
2020     130             39
2021     140             42
2022      50             27
2023      80             41
2024      85             37
2025      67             36

total: 552 scenes over 222 distinct days


---
# 5. Step two — does the scene cover the whole box?

Planet's search matches any scene that **overlaps** our box, even by a corner. But a scene
that covers 40% of the box still costs the full 100 km² minimum, and it leaves the rest of
the box as empty pixels — over exactly the water we are paying to see.

So the first gate is: **does the scene footprint contain the entire box?**

Note what this is *not*. It is not "at least 99.9% overlap". A percentage threshold has two
problems: it needs a number nobody can justify, and it still lets through scenes with a
missing corner. Instead the test is *topological* — we ask the geometry library, on a
projection where areas are true, whether the footprint **covers** the box outright. There is
no threshold to tune.

**552 scenes went in and 48 came out.** Of the 504 rejected, 332 overlapped by less than half.

In [7]:
def coverage_stats(footprint):
    """Return (fraction covered, missing square metres, does it fully contain the box).

    The third value is the actual gate. The first two are reported so that a near miss
    is visible in the output rather than just silently absent.
    """
    s = shapely_transform(to_metres, shape(footprint)).buffer(0)
    overlap = s.intersection(AOI_M).area
    return overlap / AOI_M.area, AOI_M.area - overlap, s.covers(AOI_M)

stats = {it["id"]: coverage_stats(it["geometry"]) for it in items}
df["aoi_coverage"]   = df["id"].map(lambda i: stats[i][0])
df["aoi_missing_m2"] = df["id"].map(lambda i: stats[i][1])
df["aoi_covered"]    = df["id"].map(lambda i: stats[i][2])

full   = df["aoi_covered"]
sliver = df["aoi_coverage"] < 0.5

print(f"fully contains the box : {full.sum():4d}")
print(f"partial                : {((~full) & ~sliver).sum():4d}")
print(f"less than half         : {sliver.sum():4d}")

# The scenes a percentage threshold would have waved through. Each leaves a notch of the
# box empty. Printing them is the clearest argument for the containment test.
near = df[~full & (df["aoi_coverage"] >= 0.99)]
if len(near):
    print(f"\nrejected despite covering 99% or more — this is the point of the gate:")
    for _, r in near.sort_values("aoi_coverage", ascending=False).iterrows():
        print(f"  {r['id']}   {r['aoi_coverage']:.6f} covered, {r['aoi_missing_m2']:>10,.0f} m2 missing")

before = len(df)
df = df[full].copy()
print(f"\nGATE 1: {before} -> {len(df)} scenes fully containing the box")

fully contains the box :   48
partial                :  172
less than half         :  332

rejected despite covering 99% or more — this is the point of the gate:
  20230703_182714_11_24cc   0.999905 covered,      9,483 m2 missing
  20250810_194613_51_253d   0.999820 covered,     17,999 m2 missing
  20210811_182941_82_2420   0.998502 covered,    149,529 m2 missing
  20220814_181805_13_2435   0.998419 covered,    157,783 m2 missing
  20240829_192729_36_24d2   0.997557 covered,    243,799 m2 missing
  20250728_193958_01_2506   0.996066 covered,    392,607 m2 missing
  20230726_182246_34_2460   0.995806 covered,    418,540 m2 missing
  20220728_185503_70_2481   0.994981 covered,    500,866 m2 missing
  20240716_192602_87_24cb   0.994275 covered,    571,346 m2 missing

GATE 1: 552 -> 48 scenes fully containing the box


---
# 6. Step three — how much cloud is over *our box*?

Planet ships a free per-pixel quality mask with every scene, which labels each pixel as clear,
cloud, cloud shadow, light haze, heavy haze or snow. Downloading that mask costs nothing
against the imagery allowance. So instead of trusting the whole-scene cloud number, we
download the mask for each of the 48 survivors, cut out our box, and count the fraction of
pixels labelled clear.

A scene passes if **95% or more of the box is clear**.

Two practical notes that cost real time to learn:

- **The slow part is not downloading, it is waiting.** Planet generates each mask on demand,
  and a cold one takes minutes to become available. Screening scenes one at a time took about
  4.9 minutes each — roughly four hours for the pool. The wait is idle, not work, so running
  eight at once collapsed it to **27 minutes**.
- **The scores are cached after every single scene.** A multi-hour loop will be interrupted.
  With the cache, a restart costs only the scenes that were in flight.

**48 scenes went in and 30 came out.**

In [8]:
# The screening loop is the slow, live part. Its output — one clear-fraction per scene —
# is cached in aoi_clear.json, which is what the rest of this notebook reads.
CLEAR_MIN       = 0.95
AOI_CLEAR_CACHE = WORK / "aoi_clear.json"

SCREEN_CODE = '''
import numpy as np, rasterio
from rasterio.windows import from_bounds
from rasterio.features import geometry_mask, bounds as geom_bounds
from rasterio.warp import transform_geom

UDM2_CLEAR_BAND = 1     # band 1 == 1 means "clear"; other bands are snow, shadow, haze, cloud

def aoi_clear_fraction(mask_path, aoi_geojson):
    # Fraction of the pixels inside our box that the mask calls clear.
    #
    # boundless=True pads anything outside the image with 0 (= not clear), so a scene
    # with partial coverage scores low HONESTLY instead of returning a short array that
    # would quietly misalign against the box outline.
    with rasterio.open(mask_path) as src:
        geom   = transform_geom("EPSG:4326", src.crs, aoi_geojson)
        window = from_bounds(*geom_bounds(geom), src.transform).round_offsets().round_lengths()
        clear  = src.read(UDM2_CLEAR_BAND, window=window, boundless=True, fill_value=0)
        inside = geometry_mask([geom], out_shape=clear.shape,
                               transform=src.window_transform(window), invert=True)
        return float((clear[inside] == 1).mean()) if inside.any() else float("nan")

# ...then, for each scene: activate the free ortho_udm2 asset, wait for it, download it,
# score it, delete the file, and write the score to the cache. Eight scenes at a time.
# The full loop is in the original working notebook, section 4.
'''

if not RUN_LIVE:
    would_run("downloading and scoring 48 free cloud masks (~27 minutes at 8 at a time)",
              SCREEN_CODE)

clear = json.load(open(AOI_CLEAR_CACHE))
df["aoi_clear"] = df["id"].map(lambda i: clear.get(i, float("nan")))
print(f"\n{len(clear)} cached clear-fractions read from {AOI_CLEAR_CACHE.relative_to(REPO_ROOT)}")

NOT RUN — downloading and scoring 48 free cloud masks (~27 minutes at 8 at a time)
------------------------------------------------------------------------
import numpy as np, rasterio
from rasterio.windows import from_bounds
from rasterio.features import geometry_mask, bounds as geom_bounds
from rasterio.warp import transform_geom

UDM2_CLEAR_BAND = 1     # band 1 == 1 means "clear"; other bands are snow, shadow, haze, cloud

def aoi_clear_fraction(mask_path, aoi_geojson):
    # Fraction of the pixels inside our box that the mask calls clear.
    #
    # boundless=True pads anything outside the image with 0 (= not clear), so a scene
    # with partial coverage scores low HONESTLY instead of returning a short array that
    # would quietly misalign against the box outline.
    with rasterio.open(mask_path) as src:
        geom   = transform_geom("EPSG:4326", src.crs, aoi_geojson)
        window = from_bounds(*geom_bounds(geom), src.transform).round_offsets().round_lengths()
        cle

In [9]:
# How often does the whole-scene cloud number disagree with the truth over our box?
# This is the justification for doing the work at all.
passes = df["aoi_clear"] >= CLEAR_MIN

recovered = (passes & (df["cloud_cover"] > 0.10)).sum()
rejected  = ((df["cloud_cover"] <= 0.10) & ~passes).sum()
print(f"scenes SAVED  (looked cloudy overall, but clear over our box): {recovered}")
print(f"scenes CAUGHT (looked clear overall, but cloudy over our box): {rejected}")
print(f"correlation between the two measures: "
      f"{df[['aoi_clear', 'cloud_cover']].corr().iloc[0, 1]:.2f}")

before = len(df)
df = df[passes].copy()
print(f"\nGATE 2: {before} -> {len(df)} scenes at least {CLEAR_MIN:.0%} clear over the box")

scenes SAVED  (looked cloudy overall, but clear over our box): 3
scenes CAUGHT (looked clear overall, but cloudy over our box): 4
correlation between the two measures: -0.76

GATE 2: 48 -> 30 scenes at least 95% clear over the box


---
# 7. Step four — sun glint, and looking with your eyes

**Sun glint** is sunlight bouncing off the water straight into the camera. It turns the sea
into a mirror and can hide a boat completely, or fake one. It is computed here from the sun's
position and the satellite's position — a small angle means the satellite is looking near the
sun's reflection, so more glint.

Glint is **not** used as a gate. The preview budget is generous enough that it is better to
look at a glinted scene than to throw it away unseen. It is used only to order the review
queue, so that the cleanest, least mirror-like scenes are looked at first.

The survivors are then written to `gate2_survivors.csv`. That file is what makes the human
review resumable: the table above lives only in this notebook's memory, so without the file a
reviewer would have to rebuild every gate before starting.

In [10]:
import numpy as np

# Glint angle: the angle between where the satellite is looking and where the sun's
# reflection off a flat sea would be. Small angle = looking into the glint.
sun_zenith  = np.radians(90.0 - df["sun_elevation"])
view_zenith = np.radians(df["view_angle"].abs())
delta_az    = np.radians(df["sat_azimuth"] - df["sun_azimuth"])
df["glint_angle"] = np.degrees(np.arccos(np.clip(
    np.cos(view_zenith) * np.cos(sun_zenith)
    - np.sin(view_zenith) * np.sin(sun_zenith) * np.cos(delta_az), -1, 1)))

# Cleanest first, then least glinted. This is a review ORDER, not a filter.
df = df.sort_values(["aoi_clear", "glint_angle"], ascending=False).reset_index(drop=True)

print(df["glint_angle"].describe().round(1).to_string())
print(f"\n{len(df)} scenes queued for human review")
print(df[["id", "year", "instrument", "aoi_clear", "glint_angle"]].head(8).to_string(index=False))

count    30.0
mean     35.1
std       5.4
min      24.4
25%      31.4
50%      34.9
75%      39.0
max      45.3

30 scenes queued for human review
                     id  year instrument  aoi_clear  glint_angle
20230815_182609_80_24c8  2023     PSB.SD        1.0    45.273131
20210824_181728_61_1067  2021     PS2.SD        1.0    44.750180
20210730_182659_13_2434  2021     PSB.SD        1.0    43.049759
20230714_182421_54_24b6  2023     PSB.SD        1.0    41.045202
20220816_182157_60_2457  2022     PSB.SD        1.0    39.639181
20220625_182034_44_2440  2022     PSB.SD        1.0    39.338744
20230719_182055_22_2449  2023     PSB.SD        1.0    39.017543
20240827_194346_64_2461  2024     PSB.SD        1.0    38.382815


---
# 8. Step five — previews, and the review that costs nothing

The final check before spending is a human looking at the picture. The **preview tile**
budget pays for this: Planet serves small web-map images of any scene without charging the
imagery allowance and without needing the scene to be prepared first.

At zoom level 15 the tiles are about 3.15 metres per pixel — which matches PlanetScope's own
3-metre resolution, so nothing is lost by looking at previews rather than the real thing.
Covering the whole box takes **196 tiles per scene**, so the monthly tile allowance is worth
about 500 previews. Reviewing all 30 survivors costs about 6% of it.

**The preview must cover the whole box.** Gate 1 chose scenes on containing the entire box,
so if the review only looked at the middle, scenes would be *chosen* on 100 km² and *judged*
on a fraction of it — and a boat outside the reviewed part would read as "no boat".

> ⚠️ **Not reproduced here.** The interactive review widget lives in the original working
> notebook, `contributor_folders/malachymcc/planet_folger_search_order_download.ipynb`,
> section 7. It is a click-through interface with a saved ledger, and there is no sensible
> way to re-run it from a published notebook. It is the one part of this pipeline that is
> not repeated below. The original notebook is kept in the repository for exactly this reason.

In [11]:
# The tile arithmetic — cheap, and it is what tells you whether a review is affordable.
TILE_BUDGET, TILE_ZOOM = 98249, 15

def deg2tile(lon, lat, z):
    """Longitude/latitude to a web-map tile number at zoom level z."""
    n = 2 ** z
    return (int((lon + 180.0) / 360.0 * n),
            int((1.0 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2.0 * n))

x0, y0 = deg2tile(AOI_BBOX[0], AOI_BBOX[3], TILE_ZOOM)
x1, y1 = deg2tile(AOI_BBOX[2], AOI_BBOX[1], TILE_ZOOM)
per_scene = (x1 - x0 + 1) * (y1 - y0 + 1)
metres_per_pixel = (360 / 2 ** TILE_ZOOM) * 111_320 * math.cos(math.radians(HYD_LAT)) / 256

print(f"tiles to cover the whole box : {x1-x0+1} x {y1-y0+1} = {per_scene}")
print(f"ground resolution at zoom {TILE_ZOOM} : {metres_per_pixel:.2f} m per pixel"
      f"   (PlanetScope itself is 3 m)")
print(f"previews the monthly budget affords : {TILE_BUDGET // per_scene:,}")
print(f"reviewing all {len(df)} survivors costs : {len(df)*per_scene:,} tiles"
      f"  ({len(df)*per_scene/TILE_BUDGET:.1%} of the month)")

ledger = json.load(open(WORK / "tile_ledger.json"))
print(f"\nactually spent in {ledger['month']}: {ledger['used']:,} of {TILE_BUDGET:,} tiles")
show_folder(WORK / "tile_previews", "the 30 preview images, one per surviving scene")

tiles to cover the whole box : 14 x 14 = 196
ground resolution at zoom 15 : 3.15 m per pixel   (PlanetScope itself is 3 m)
previews the monthly budget affords : 501
reviewing all 30 survivors costs : 5,880 tiles  (6.0% of the month)

actually spent in 2026-08: 12,372 of 98,249 tiles


   absolute path: /home/jovyan/ohw26_proj_BoatPhone/contributor_folders/malachymcc/planet_folger/tile_previews


---
# 9. Step six — check the microphone was recording, *before* spending

This is the cheapest possible way to avoid wasting a photograph, and it is easy to forget.

A satellite image is only useful to this project if there is a **sound recording from the same
moment** to compare it against. The hydrophone was not running continuously for six years: it
has outages, and it was not deployed at all in 2026.

So before ordering anything, every surviving scene is checked against the recorded outages in
`docs/derived/hydrophone_gaps.md`, which the acoustic pipeline produces.

**All 30 survivors cleared.** The closest any of them came to an outage was 146 hours — just
over six days.

> 🛰️ **HANDOFF — answered.** The acoustic notebook flags *"do not order 2026 imagery"*: any
> scene taken after 14 March 2026 has no sound to compare against, so the order would be
> unrecoverably wasted. This pipeline never had that risk, because the search was restricted
> to 2020–2025 from the start. The check below is the standing guard for anyone extending it.

> ⚠️ **Re-run this if the calendar is refined.** The outage list is derived from Planet's
> *listing* of which sound files exist, not from having downloaded them. A scene can clear
> this check and still sit on a dropout shorter than the listing resolution.

In [12]:
# The check itself is a few lines. The point is that it is done BEFORE the spending gate.
GAPS_DOC = REPO_ROOT / "docs" / "derived" / "hydrophone_gaps.md"

if GAPS_DOC.exists():
    print(f"outage record: {GAPS_DOC.relative_to(REPO_ROOT)}")
    print(GAPS_DOC.read_text()[:900])
else:
    print(f"NOT FOUND: {GAPS_DOC}")
    print("This file is produced by the acoustic pipeline (see acoustic_pipeline.ipynb,")
    print("section on the uptime calendar). Do not order imagery without it.")

outage record: docs/derived/hydrophone_gaps.md
# Hydrophone uptime and no-data spans -- Folger Deep (ICLISTENHF1266)

**Deliverable O1, human-readable half. For Malachy (Planet acquisition).**
The machine-readable half is `data/derived/hydrophone_uptime.csv`, which is gitignored and
therefore does not travel; this file does.

---

## The one line that matters

**These date ranges have no hydrophone data -- do not spend Planet quota on them:**

| Span (UTC, end EXCLUSIVE) | Length | What it is |
|---|---|---|
| `2026-05-01` -> `2026-10-01` | 153 days (the entire 2026 season) | ICLISTENHF1266's deployment ENDED `2026-03-14T21:37:20Z`. ONC reports no deployment of this device at Folger during the 2026 season. The archive request for the span returns ONC's *API Error 127: A device with category HYDROPHONE was deployed at location FGPD but not during the provided time range* -- under decision 0007 that is a **measured zero**, i.e. ONC pos


---
# 10. Step seven — choosing the 26, and the question we never answered

Three scenes were removed from the 30 by hand, each for a stated reason:

| Scene | Why it was dropped |
|---|---|
| `20210616_183140_94_2436` | Taken **30 seconds** after another survivor, by the same satellite. That is one observation, not two. The marginally clearer of the pair was kept. |
| `20230618_182410_36_24ab` | The thinnest survivor by clear fraction (0.9737). Cloud shadow and cloud edges are exactly what fake a boat for an infrared brightness test. |
| `20240701_193748_00_2482` | Second thinnest (0.9747), same reason. |

Note what was **not** done: scenes were not de-duplicated by day. Two of the survivors are 25
minutes apart on different satellites, and a boat moves a long way in 25 minutes — those are
two genuinely independent observations. Only the 30-second pair is redundant.

## Why 26 and not 30

Because we still do not know what Planet actually charges for.

The account's usage meter never moved off zero, even after a scene had been ordered *and*
downloaded, so it could not answer the question. What we do know is measured from the file
that arrived: **the delivered image is 105.82 km², not the 99.800 km² we asked to cut out.**
The cut-out is delivered in a national grid whose north is rotated about 1.7° from ours, and
the bounding rectangle of a rotated 10 km square is bigger than the square.

So the batch size was chosen to survive **either** billing basis:

| If Planet charges... | 26 scenes + 2 controls costs | Against the 3,000 km² allowance |
|---|---|---|
| the 100 km² minimum | 2,800 km² | fits |
| the delivered 105.82 km² rectangle | 2,963 km² | fits |
| *(27 scenes on the delivered basis)* | *3,068 km²* | **over by 68** |

There is no next month to recover in — the allowance resets on the calendar and the project
ends first.

> ⚠️ **Open question for whoever runs this next.** Order **one** scene, read the usage meter
> before and after, and settle it. One scene costs a thirtieth of a month and turns this
> paragraph into a fact.

In [13]:
survivors = pd.read_csv(WORK / "gate2_survivors.csv")

DROP_IDS = {
    "20210616_183140_94_2436",   # 30 s after another survivor, same satellite
    "20230618_182410_36_24ab",   # thinnest by clear fraction
    "20240701_193748_00_2482",   # second thinnest
}
# The two scenes also bought as Planet's own colour render, as a controlled comparison:
# the SAME acquisition in two forms, so any difference is the rendering, not the ocean.
PAIRED_VISUAL_IDS = ["20200730_192942_71_1059", "20250630_193824_56_24db"]

missing = DROP_IDS - set(survivors["id"])
assert not missing, f"a scene on the drop list is not in the survivor pool: {sorted(missing)}"

batch = (survivors[~survivors["id"].isin(DROP_IDS)]
         .sort_values("aoi_clear", ascending=False)      # cleanest first
         .head(26))

missing = set(PAIRED_VISUAL_IDS) - set(batch["id"])
assert not missing, f"a paired control is not in the batch — it would be billed unused: {sorted(missing)}"

DELIVERED_BBOX_KM2 = 105.8212     # MEASURED from the delivered file, not assumed
slots = len(batch) + len(PAIRED_VISUAL_IDS)
print(f"{len(survivors)} survivors -> {len(survivors)-len(DROP_IDS)} after the drop list"
      f" -> {len(batch)} in the batch")
print(f"clear-fraction range in the batch: {batch['aoi_clear'].min():.3f} to {batch['aoi_clear'].max():.3f}\n")
for label, per_scene_km2 in (("the 100 km2 minimum   ", CHARGE_PER_SCENE),
                             ("the delivered rectangle", DELIVERED_BBOX_KM2)):
    total = slots * per_scene_km2
    print(f"  if charged on {label}: {total:7.1f} of {MONTHLY_QUOTA:.0f} km2"
          f"   ({MONTHLY_QUOTA - total:+.1f} to spare)")

30 survivors -> 27 after the drop list -> 26 in the batch
clear-fraction range in the batch: 0.985 to 1.000

  if charged on the 100 km2 minimum   :  2800.0 of 3000 km2   (+200.0 to spare)
  if charged on the delivered rectangle:  2963.0 of 3000 km2   (+37.0 to spare)


---
# 11. Step eight — the order, behind two gates

This is the only cell in the notebook that spends anything.

Two details matter:

- **Always cut the scene down to the box.** A full PlanetScope scene is about 637 km²; our box
  is 100. Ordering without the cut would cost six times as much for the same science.
- **Order the right product.** We need the near-infrared band — light just past what the eye
  can see, in which water is very dark and boats are bright. Only one product bundle carries
  it for these satellites. The quality mask arrives inside the same bundle and, importantly,
  **does not draw a second charge**: one bundle, one delivery, one charge.

The two paired colour-render scenes are a separate order, because one order carries one
product type.

In [14]:
ORDER_CODE = '''
from planet import Auth, Session, order_request, reporting

BUNDLE        = "analytic_sr_udm2"   # the near-infrared carrier + its quality mask
PAIRED_BUNDLE = "visual"             # Planet's own colour render, 2 control scenes only

async def place_and_download(ids, name, bundle=BUNDLE):
    out = WORK / "downloads" / name
    out.mkdir(parents=True, exist_ok=True)
    async with Session(auth=Auth.from_key(API_KEY)) as sess:
        client  = sess.client("orders")
        request = order_request.build_request(
            name=name,
            products=[order_request.product(item_ids=ids, product_bundle=bundle,
                                            item_type="PSScene")],
            tools=[order_request.clip_tool(aoi=aoi)],   # NEVER skip this: 6x saving
        )
        with reporting.StateBar(state="creating") as bar:
            order = await client.create_order(request)
            bar.update(state="created", order_id=order["id"])
            await client.wait(order["id"], callback=bar.update_state, max_attempts=0)
        await client.download_order(order["id"], directory=out, progress_bar=True)
    return order["id"], out

order_id, folder = await place_and_download(list(batch["id"]), "folger_202608")
visual_id, _     = await place_and_download(PAIRED_VISUAL_IDS, "folger_202608_visual",
                                            bundle=PAIRED_BUNDLE)
'''

if ALLOW_SPENDING:
    print("ALLOW_SPENDING is True. Paste the code below into a cell of its own and run it —")
    print("it needs `await` at notebook top level. THIS SPENDS THE MONTHLY ALLOWANCE.")
    print(ORDER_CODE)
else:
    would_run("PLACING THE ORDER — this permanently spends the monthly imagery allowance",
              ORDER_CODE)
    print("\nSet ALLOW_SPENDING = True in the setup cell only when you mean it.")

NOT RUN — PLACING THE ORDER — this permanently spends the monthly imagery allowance
------------------------------------------------------------------------
from planet import Auth, Session, order_request, reporting

BUNDLE        = "analytic_sr_udm2"   # the near-infrared carrier + its quality mask
PAIRED_BUNDLE = "visual"             # Planet's own colour render, 2 control scenes only

async def place_and_download(ids, name, bundle=BUNDLE):
    out = WORK / "downloads" / name
    out.mkdir(parents=True, exist_ok=True)
    async with Session(auth=Auth.from_key(API_KEY)) as sess:
        client  = sess.client("orders")
        request = order_request.build_request(
            name=name,
            products=[order_request.product(item_ids=ids, product_bundle=bundle,
                                            item_type="PSScene")],
            tools=[order_request.clip_tool(aoi=aoi)],   # NEVER skip this: 6x saving
        )
        with reporting.StateBar(state="creating") as bar:
  

---
# 12. What arrived

Twenty-six scenes, delivered in three forms, and a manifest describing all of them.

The **manifest is the join key back to the sound**. It carries each scene's exact acquisition
instant in UTC, which is what the acoustic pipeline matches its recordings against. A
PlanetScope scene is a single snapshot, so every boat in a scene shares one timestamp.

In [15]:
manifest = pd.read_csv(WORK / "manifest_folger_202608.csv")
print(f"{len(manifest)} scenes in the manifest\n")
print(manifest[["id", "acq_time_utc", "instrument", "aoi_clear", "hydrophone_gap_free",
                "hours_to_nearest_outage"]].to_string(index=False))

26 scenes in the manifest

                     id                acq_time_utc instrument  aoi_clear  hydrophone_gap_free  hours_to_nearest_outage
20200730_192942_71_1059 2020-07-30T19:29:42.712659Z     PS2.SD   1.000000                 True                  17700.4
20200806_183328_89_2206 2020-08-06T18:33:28.895946Z     PSB.SD   1.000000                 True                  17533.4
20210601_191849_05_2412 2021-06-01T19:18:49.050614Z     PSB.SD   1.000000                 True                  10356.6
20210616_183111_05_2448 2021-06-16T18:31:11.057048Z     PSB.SD   0.997986                 True                   9997.4
20210626_182926_29_2457 2021-06-26T18:29:26.294082Z     PSB.SD   1.000000                 True                   9757.4
20210730_182659_13_2434 2021-07-30T18:26:59.136637Z     PSB.SD   1.000000                 True                   8941.5
20210812_191832_32_2406 2021-08-12T19:18:32.325059Z     PSB.SD   1.000000                 True                   8628.6
20210812_1943

In [16]:
# Where the pixels actually are. They are too large for git, so they live on the hub's
# shared storage. Both paths are printed; whichever exists on your machine is the one to use.
SHARED = pathlib.Path("/home/jovyan/shared-public/boatphone_shared/planet_folger")

print("Delivered imagery, three forms of the same 26 acquisitions:\n")
for folder, what in (
        ("downloads/folger_202608_toa",    "as the satellite measured it, before haze removal"),
        ("downloads/folger_202608",        "after software removes the haze"),
        ("downloads/folger_202608_visual", "Planet's own colour render — 2 control scenes only"),
        ("tile_previews",                  "the 30 preview images used for the human review")):
    p = SHARED / folder
    mark = "present" if p.exists() else "NOT on this machine"
    print(f"  {folder:<34} {what}\n      {p}   [{mark}]")

if SHARED.exists():
    show_folder(SHARED, "shared imagery folder (read its README first)")

Delivered imagery, three forms of the same 26 acquisitions:

  downloads/folger_202608_toa        as the satellite measured it, before haze removal
      /home/jovyan/shared-public/boatphone_shared/planet_folger/downloads/folger_202608_toa   [present]
  downloads/folger_202608            after software removes the haze
      /home/jovyan/shared-public/boatphone_shared/planet_folger/downloads/folger_202608   [present]
  downloads/folger_202608_visual     Planet's own colour render — 2 control scenes only
      /home/jovyan/shared-public/boatphone_shared/planet_folger/downloads/folger_202608_visual   [present]
  tile_previews                      the 30 preview images used for the human review
      /home/jovyan/shared-public/boatphone_shared/planet_folger/tile_previews   [present]


   absolute path: /home/jovyan/shared-public/boatphone_shared/planet_folger


> ⚠️ **The difference between those first two folders is the single most important fact on
> this side of the project**, and it is not obvious. One is the brightness the satellite
> actually recorded; the other is the same picture after software has tried to subtract the
> atmosphere. Over water, the atmosphere is *most of the signal* — and the boat detector works
> on one and fails on the other. `vessel_detection_pipeline.ipynb` section 5 is entirely about
> this. Do not treat them as interchangeable.

---
# 13. What is still wrong, in priority order

Honest list. A future scientist should read this before extending anything.

### 1. A third of the season was never searched
The search ran June to August. The project's shared definition of the field season, in
`boatphone/config.py`, is **May to September**. All 552 candidates are therefore from months
06, 07 and 08, and roughly a third of the eligible pool was never looked at.

This is a straightforward fix — change the month range and re-run the search, which is free —
but it was deliberately not done in-session, because editing the working notebook while it was
open in the browser had already cost a full day's work once. **Searching May and September is
also the first fallback** if the usable set ever drops below 30, ahead of accepting scenes that
only partly cover the box: a May scene still covers the whole box, whereas a partial one makes
the search area change between observations. The catch is that a lower sun in May and September
means more glint, and three of the four hydrophone outages fall in September 2023.

### 2. The billing basis is still a guess
See section 10. One scene, one before-and-after reading of the usage meter, and it is settled.

### 3. The hydrophone coordinate is the box centre, not the instrument

> 🛰️ **HANDOFF — still open.** `optical.HYDROPHONE_LONLAT` is the centre of our study box,
> not confirmed instrument metadata. A claimed position from the observatory sits **272 metres
> away**, which at a range of 0.84 km is a **32% distance error** — and every distance in the
> detection outputs is measured from this point. Worse, the observatory records the position
> *per deployment*, so no single coordinate can be right for all six years. Settling it needs
> `getDeployments(deviceCode="ICLISTENHF1266")` run with an account token.

### 4. The human review is unfinished
Gate 3 — a person looking at all 30 previews for boats — was never completed. It turned out not
to block anything, because empty scenes are useful too (they measure the background noise), so
the order went ahead on the two hard gates alone. But the previews are on disk and the review
would still be worth doing.

---
# Where to go next

| Notebook | What it covers |
|---|---|
| `vessel_detection_pipeline.ipynb` | Finding boats in these 26 scenes: everything that was tried, what failed and why, and the detector that shipped. |
| `acoustic_pipeline.ipynb` | The underwater microphone half — recordings, loudness, and counting boat passages by sound. |

**Original working notebook:** `contributor_folders/malachymcc/planet_folger_search_order_download.ipynb`
holds the live API code, the interactive review widget and the running tile ledger. It is
messier than this one by design; this notebook is the readable record, that one is the
instrument.

**Retired approaches** are in `superseded/optical/`, which is kept out of git. Its README
lists what each file was and the measurement that retired it.